In [ ]:
%matplotlib widget
import matplotlib.pyplot as plt
import nest_asyncio

nest_asyncio.apply()  # used in notebook

from phyling.ble import NanoPhyling, MiniPhyling

## NanoPhyling

In [ ]:
device = NanoPhyling(
    ble_name="NanoPhyling_42",
    config={
        "rate": 200,
        "data": ["acc_x", "acc_y", "acc_z", "gyro_x", "gyro_y", "gyro_z", "mag_x", "mag_y", "mag_z"],
    }
)

## MiniPhyling

In [ ]:
device = MiniPhyling(
    ble_name="MiniPhyling_42",
    # La config est automatiquement lue depuis le Mini-Phyling. Pour la changer, il faut modifier le Mini-Phyling en mode USB.
)

## Calibration

`tare()` enregistre 5 secondes de données immobiles et calcule un offset pour chaque colonne.
La calibration est appliquée automatiquement dans `get_df()` : `coef * (val + offset)`.
Le df existant n'est **pas** écrasé.

In [ ]:
# Tare : calcule l'offset de chaque colonne sur 5s de données immobiles
device.tare(["gyro_x", "gyro_y", "gyro_z"])

In [ ]:
# Voir / modifier la calibration manuellement
device.get_calib()

In [ ]:
# Reset la calibration (vide le dict, pas de calibration)
device.reset_calib()

In [ ]:
# Remplacer entièrement la calibration
device.set_calib({
    "gyro_z": {"coef": 1, "offset": -0.012},
})

In [ ]:
# Merger une calibration partielle (garde les autres colonnes)
device.update_calib({
    "acc_z": {"coef": 1.02, "offset": 0},
})

## Acquisition

In [ ]:
# Vous pouvez kill la cell pendant l'exécution pour arrêter l'enregistrement
device.run(duration=30)

In [ ]:
# La calibration est appliquée ici (coef * (val + offset))
df = device.get_df()
df.head()

In [ ]:
plt.figure()
plt.plot(df["T"], df["acc_x"], label="acc_x")
plt.plot(df["T"], df["acc_y"], label="acc_y")
plt.plot(df["T"], df["acc_z"], label="acc_z")
plt.legend()

plt.figure()
plt.plot(df["T"], df["gyro_x"], label="gyro_x")
plt.plot(df["T"], df["gyro_y"], label="gyro_y")
plt.plot(df["T"], df["gyro_z"], label="gyro_z")
plt.legend()

In [ ]:
# Enregistrer les données dans un CSV
# df.to_csv("example_data.csv", index=False)